In [1]:
from huggingface_hub import snapshot_download

snapshot_download(repo_id="MrDragonFox/Elise", repo_type="dataset", local_dir="./Elise")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 3 files: 100%|██████████| 3/3 [00:02<00:00,  1.16it/s]


'/home/ubuntu/Elise'

In [2]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [3]:
files = glob('Elise/data/*.parquet')
files

['Elise/data/train-00000-of-00001.parquet']

In [4]:
df = pd.read_parquet(files[0])
df

,audio,text
0,{'bytes': b'RIFF\xcc\xba\x06\x00WAVEfmt \x10\x...,"Please have mercy on my dainty, frail body. Yo..."
1,{'bytes': b'RIFF\xcc\xba\x06\x00WAVEfmt \x10\x...,"The borrowers. Yeah, beneath the floorboards. ..."
2,{'bytes': b'RIFF\xb8\xdb\x04\x00WAVEfmt \x10\x...,Keep me alive from his victims. And he had ple...
3,{'bytes': b'RIFF\xe62\x04\x00WAVEfmt \x10\x00\...,"Can I get my scritchies now? Oh, but I thought..."
4,{'bytes': b'RIFF|\x9b\x03\x00WAVEfmt \x10\x00\...,"I mean, what kind of crazy person would just b..."
...,...,...
1190,{'bytes': b'RIFFF\x81\x05\x00WAVEfmt \x10\x00\...,"Hun, I don't like seeing you like this. You're..."
1191,{'bytes': b'RIFF\x80\xd9\x03\x00WAVEfmt \x10\x...,You got something to say about that? <giggles>...
1192,{'bytes': b'RIFF\xf6*\x05\x00WAVEfmt \x10\x00\...,They made some cute stuff for me. It was fucki...
1193,{'bytes': b'RIFF\x8at\x04\x00WAVEfmt \x10\x00\...,I had a look around her apartment after you le...


In [7]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in tqdm(files):
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in range(len(df)):
            t = df['text'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"MrDragonFox_Elise"
            })
        
    return data

In [8]:
data = loop((files[:1], 0))

100%|██████████| 1/1 [01:07<00:00, 67.57s/it]


In [9]:
len(data)

1194

In [10]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'Elise_audio/Elise-data-train-00000-of-00001_0.mp3',
 'text': 'Please have mercy on my dainty, frail body. Your coils are so strong and powerful, and I am powerless to resist.',
 'speaker': 'MrDragonFox_Elise'}

In [11]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'Elise')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 843.58ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████| 82.9kB / 82.9kB,  415kB/s  
Processing Files (1 / 1): 100%|██████████| 82.9kB / 82.9kB,  208kB/s  
New Data Upload: 100%|██████████| 82.9kB / 82.9kB,  208kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.39 shards/s]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/f5dfbf982c41dcf534d257c6d08eaf62f812b747', commit_message='Upload dataset', commit_description='', oid='f5dfbf982c41dcf534d257c6d08eaf62f812b747', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [12]:
audio_files = [d['audio_filename'] for d in data]

with open('Elise-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [14]:
folders = glob('Elise_audio*')
folders = [f for f in folders if '.zip' not in f]
for f in folders:
    print(f)
    os.system(f'zip -rq {f}.zip {f}')

Elise_audio
Elise_audio_neucodec


In [15]:
from huggingface_hub import HfApi
api = HfApi()

for f in glob('Elise_audio*.zip'):
    api.upload_file(
        path_or_fileobj=f,
        path_in_repo=f,
        repo_id="malaysia-ai/Multilingual-TTS",
        repo_type="dataset",
    )

Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████| 1.54MB / 1.54MB,   ???B/s  
Processing Files (1 / 1): 100%|██████████| 1.54MB / 1.54MB,  0.00B/s  
New Data Upload: 100%|██████████| 1.54MB / 1.54MB,  0.00B/s  
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1): 100%|█████████▉| 49.9MB / 50.0MB,   ???B/s  
Processing Files (1 / 1): 100%|██████████| 50.0MB / 50.0MB, 86.5kB/s  
Processing Files (1 / 1): 100%|██████████| 50.0MB / 50.0MB, 74.1kB/s  
New Data Upload: 100%|██████████| 50.0MB / 50.0MB, 74.1kB/s  
